In [1]:
import pandas as pd
import numpy as np
import glob

from collections import defaultdict
from typing import List
from metrics import assert_valid_prob, assert_same_exprs, compute_wasserstein_distance
from utils_io import read_json
from utils_latex import *
from default_vars import UNCERTAINTY_EXPRESSIONS

EXCLUDE_MODELS = [
    "full__lmsys__vicuna-13b-v1.5",
    "full__mistralai__Mistral-7B-Instruct-v0.2",
    "sampling__allenai__OLMo-7B-Instruct","sampling__google__gemma-1.1-2b-it"
]

In this notebook, we want to compute the difference between the empirical distributions when estimated using
greedy decoding vs probabilistic decoding.

In [2]:
_non_verifiable_results_ws = []
for n_shots in (0, 2):
    model_filepaths = sorted(glob.glob(f"../../results/greedy/all/non_verifiable/models-{n_shots}shot/*gpt*_normalized.csv"))
        
    for greedy_fp in model_filepaths:
        print("Processing", greedy_fp)
        greedy_df = pd.read_csv(greedy_fp, index_col=0)
        prob_df = pd.read_csv(greedy_fp.replace("greedy", "probabilistic"), index_col=0)
        dist = compute_wasserstein_distance(greedy_df, prob_df, uncertainty_expressions=UNCERTAINTY_EXPRESSIONS)
        dist["n_shots"] = n_shots
        dist["setting"] = "non-verifiable"
        
        model_name = greedy_fp.rpartition("shot/")[-1].rpartition("_normalized")[0]
        dist["model"] = model_name
        _non_verifiable_results_ws.append(dist)

_non_verifiable_results_ws = pd.concat(_non_verifiable_results_ws, axis=0).reset_index(drop=True)
_nv_ws_dist_avg = _non_verifiable_results_ws.drop(["setting", "uncertainty_expression"], axis=1)\
     .groupby(["n_shots", "model"])\
     .mean().reset_index().pivot(index="model", columns="n_shots", values="distance")
_nv_ws_dist_avg

Processing ../../results/greedy/all/non_verifiable/models-0shot/gpt-3.5-turbo-0125_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-0shot/gpt-4-turbo-2024-04-09_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-0shot/gpt-4o-2024-05-13_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-2shot/gpt-3.5-turbo-0125_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-2shot/gpt-4-turbo-2024-04-09_normalized.csv
Processing ../../results/greedy/all/non_verifiable/models-2shot/gpt-4o-2024-05-13_normalized.csv


n_shots,0,2
model,,
gpt-3.5-turbo-0125,3.280135,3.316445
gpt-4-turbo-2024-04-09,0.305923,0.215723
gpt-4o-2024-05-13,19.876309,15.375068


### Verifiable

In [3]:
_verifiable_results_ws = []
for n_shots in (0, 2):
    model_filepaths = sorted(glob.glob(f"../../results/greedy/all/verifiable/models-{n_shots}shot/*gpt*_normalized.csv"))
        
    for greedy_fp in model_filepaths:
        print("Processing", greedy_fp)
        greedy_df = pd.read_csv(greedy_fp, index_col=0)
        prob_df = pd.read_csv(greedy_fp.replace("greedy", "probabilistic"), index_col=0)
        dist = compute_wasserstein_distance(greedy_df, prob_df, uncertainty_expressions=UNCERTAINTY_EXPRESSIONS)
        dist["n_shots"] = n_shots
        dist["setting"] = "verifiable"
        
        model_name = greedy_fp.rpartition("shot/")[-1].rpartition("_normalized")[0]
        dist["model"] = model_name
        _verifiable_results_ws.append(dist)

_verifiable_results_ws = pd.concat(_verifiable_results_ws, axis=0).reset_index(drop=True)
_v_ws_dist_avg = _verifiable_results_ws.drop(["setting", "uncertainty_expression"], axis=1)\
     .groupby(["n_shots", "model"])\
     .mean().reset_index().pivot(index="model", columns="n_shots", values="distance")
_v_ws_dist_avg

Processing ../../results/greedy/all/verifiable/models-0shot/gpt-3.5-turbo-0125_normalized.csv
Processing ../../results/greedy/all/verifiable/models-0shot/gpt-4-turbo-2024-04-09_normalized.csv
Processing ../../results/greedy/all/verifiable/models-0shot/gpt-4o-2024-05-13_normalized.csv
Processing ../../results/greedy/all/verifiable/models-2shot/gpt-3.5-turbo-0125_normalized.csv
Processing ../../results/greedy/all/verifiable/models-2shot/gpt-4-turbo-2024-04-09_normalized.csv
Processing ../../results/greedy/all/verifiable/models-2shot/gpt-4o-2024-05-13_normalized.csv


n_shots,0,2
model,,
gpt-3.5-turbo-0125,3.398651,2.747937
gpt-4-turbo-2024-04-09,0.518217,0.841152
gpt-4o-2024-05-13,15.506152,19.861464


### Verifiable (AI2 Arc)

In [4]:
_verifiable_ai2arc_easy_results_ws = []

for n_shots in (0, 2):
    model_filepaths = sorted(glob.glob(f"../../results/greedy/all/verifiable-ai2arc-easy/models-{n_shots}shot/*gpt*_normalized.csv"))
        
    for greedy_fp in model_filepaths:
        print("Processing", greedy_fp)
        greedy_df = pd.read_csv(greedy_fp, index_col=0)
        prob_df = pd.read_csv(greedy_fp.replace("greedy", "probabilistic"), index_col=0)
        dist = compute_wasserstein_distance(greedy_df, prob_df, uncertainty_expressions=UNCERTAINTY_EXPRESSIONS)
        dist["n_shots"] = n_shots
        dist["setting"] = "verifiable"
        
        model_name = greedy_fp.rpartition("shot/")[-1].rpartition("_normalized")[0]
        dist["model"] = model_name
        _verifiable_ai2arc_easy_results_ws.append(dist)

_verifiable_ai2arc_easy_results_ws = pd.concat(_verifiable_ai2arc_easy_results_ws, axis=0).reset_index(drop=True)
_vai2arc_easy_ws_dist_avg = _verifiable_ai2arc_easy_results_ws.drop(["setting", "uncertainty_expression"], axis=1)\
     .groupby(["n_shots", "model"])\
     .mean().reset_index().pivot(index="model", columns="n_shots", values="distance")
_vai2arc_easy_ws_dist_avg

Processing ../../results/greedy/all/verifiable-ai2arc-easy/models-2shot/gpt-3.5-turbo-0125_normalized.csv
Processing ../../results/greedy/all/verifiable-ai2arc-easy/models-2shot/gpt-4-turbo-2024-04-09_normalized.csv
Processing ../../results/greedy/all/verifiable-ai2arc-easy/models-2shot/gpt-4o-2024-05-13_normalized.csv


n_shots,2
model,
gpt-3.5-turbo-0125,2.877336
gpt-4-turbo-2024-04-09,0.487571
gpt-4o-2024-05-13,20.699033


In [5]:
_verifiable_ai2arc_chall_results_ws = []

for n_shots in (0, 2):
    model_filepaths = sorted(glob.glob(f"../../results/greedy/all/verifiable-ai2arc-challenge/models-{n_shots}shot/*gpt*_normalized.csv"))
        
    for greedy_fp in model_filepaths:
        print("Processing", greedy_fp)
        greedy_df = pd.read_csv(greedy_fp, index_col=0)
        prob_df = pd.read_csv(greedy_fp.replace("greedy", "probabilistic"), index_col=0)
        dist = compute_wasserstein_distance(greedy_df, prob_df, uncertainty_expressions=UNCERTAINTY_EXPRESSIONS)
        dist["n_shots"] = n_shots
        dist["setting"] = "verifiable"
        
        model_name = greedy_fp.rpartition("shot/")[-1].rpartition("_normalized")[0]
        dist["model"] = model_name
        _verifiable_ai2arc_chall_results_ws.append(dist)

_verifiable_ai2arc_chall_results_ws = pd.concat(_verifiable_ai2arc_chall_results_ws, axis=0).reset_index(drop=True)
_vai2arc_chall_ws_dist_avg = _verifiable_ai2arc_chall_results_ws.drop(["setting", "uncertainty_expression"], axis=1)\
     .groupby(["n_shots", "model"])\
     .mean().reset_index().pivot(index="model", columns="n_shots", values="distance")
_vai2arc_chall_ws_dist_avg

Processing ../../results/greedy/all/verifiable-ai2arc-challenge/models-2shot/gpt-3.5-turbo-0125_normalized.csv
Processing ../../results/greedy/all/verifiable-ai2arc-challenge/models-2shot/gpt-4-turbo-2024-04-09_normalized.csv
Processing ../../results/greedy/all/verifiable-ai2arc-challenge/models-2shot/gpt-4o-2024-05-13_normalized.csv


n_shots,2
model,
gpt-3.5-turbo-0125,3.075805
gpt-4-turbo-2024-04-09,0.777213
gpt-4o-2024-05-13,20.967463
